# DeepHit Backtest Grid Search Analysis

This notebook is focused on selecting the final DeepHit strategy settings from the 3D backtest grid: `max_toxic_probability` x `min_fill_probability` x `horizon_index`.

The workflow is intentionally compact:

1. Load one ticker/model grid-search CSV.
2. Normalize old and new report column names.
3. Rank configurations with optional participation/failure constraints.
4. Inspect per-horizon heatmaps and trade-offs.
5. Export the selected strategy settings.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.parent.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
# ---- User configuration ----
TICKER = "AAPL"
MODEL_TYPE = "gru_transformer"

# output root from scripts/deephit_bt_grid_search.py.
OUTPUT_ROOT = PROJECT_DIR / "reports" / "deephit_bt_grid_search_core_plus_0_csv"

# Optional explicit file override. Leave as None for default path discovery.
RESULT_PATH_OVERRIDE = None

# Optional runtime NPZ for mapping horizon_index -> seconds. Leave as None for default discovery.
RUNTIME_NPZ_PATH_OVERRIDE = None
DATA_ROOT = PROJECT_DIR / "data"
DATA_START_DATE = "2025-10-01"
DATA_END_DATE = "2026-01-01"

# Selection objective. Lower is better for IS columns.
OBJECTIVE_COL = "toxic_cost_mean_bps"  # alternatives: "mean_is_bps", "median_is_bps", "time_weighted_mean_is_bps", "toxic_cost_mean_bps"

# Optional minimum/maximum constraints for picking final settings.
MIN_SUBMITTED_RATE = 0
MAX_METRIC_FAILED_RATE = 0.01
MIN_FILL_RATE = 0

TOP_N = 20

In [ ]:
COLUMN_ALIASES = {
    "submitted_orders": "submitted",
    "skipped_orders": "skipped",
    "filled_orders": "filled",
    "unfilled_orders": "unfilled",
    "canceled_orders": "canceled",
    "metric_ok_orders": "metric_ok",
    "metric_failed_orders": "metric_failed",
    "mean_implementation_shortfall_bps": "mean_is_bps",
    "median_implementation_shortfall_bps": "median_is_bps",
    "total_implementation_shortfall": "total_is",
    "total_implementation_shortfall_raw": "total_is_raw",
}

REQUIRED_GRID_COLS = [
    "max_toxic_probability",
    "min_fill_probability",
    "horizon_index",
]


def compact_date(date: str) -> str:
    return str(date).replace("-", "")


def default_result_path(ticker: str, model_type: str) -> Path:
    return OUTPUT_ROOT / ticker.upper() / model_type / "deephit_threshold_grid_search.csv"


def legacy_result_candidates(ticker: str, model_type: str) -> list[Path]:
    return [
        default_result_path(ticker, model_type),
        PROJECT_DIR / "reports" / f"{model_type}_grid_search.csv",
        PROJECT_DIR / "reports" / "deephit_threshold_grid_search.csv",
        PROJECT_DIR / "reports" / "deephit_threshold_grid_search_old.csv",
    ]


def resolve_result_path(ticker: str, model_type: str, override=None) -> Path:
    if override is not None:
        return Path(override)
    for path in legacy_result_candidates(ticker, model_type):
        if path.exists():
            return path
    return default_result_path(ticker, model_type)


def normalize_grid_frame(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.rename(columns={k: v for k, v in COLUMN_ALIASES.items() if k in raw.columns}).copy()
    missing = [c for c in REQUIRED_GRID_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Grid search CSV is missing required columns: {missing}")

    numeric_cols = [
        "orders", "submitted", "skipped", "filled", "unfilled", "canceled",
        "metric_ok", "metric_failed", "fill_rate", "mean_is_bps", "median_is_bps",
        "time_weighted_mean_is_bps", "opportunity_cost_mean_bps", "toxic_cost_mean_bps",
        "max_toxic_probability", "min_fill_probability", "horizon_index",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "orders" in df.columns:
        if "submitted" in df.columns:
            df["submitted_rate"] = df["submitted"] / df["orders"]
        if "skipped" in df.columns:
            df["skipped_rate"] = df["skipped"] / df["orders"]
        if "canceled" in df.columns:
            df["canceled_rate"] = df["canceled"] / df["orders"]
        if "metric_failed" in df.columns:
            df["metric_failed_rate"] = df["metric_failed"] / df["orders"]

    df["horizon_index"] = df["horizon_index"].astype(int)
    sort_cols = ["horizon_index", "max_toxic_probability", "min_fill_probability"]
    return df.sort_values(sort_cols).reset_index(drop=True)


def resolve_runtime_npz_path(ticker: str, override=None) -> Path | None:
    if override is not None:
        path = Path(override)
        return path if path.exists() else None
    stem = f"labeled_dataset_XNAS_ITCH_{ticker.upper()}_mbo_{compact_date(DATA_START_DATE)}_{compact_date(DATA_END_DATE)}"
    candidates = [
        DATA_ROOT / "datasets" / f"{stem}_dynamic_preprocessed.npz",
        Path("/ocean/projects/cis260122p/shared/data/datasets") / f"{stem}_dynamic_preprocessed.npz",
    ]
    return next((p for p in candidates if p.exists()), None)


def add_horizon_seconds(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    out = df.copy()
    npz_path = resolve_runtime_npz_path(ticker, RUNTIME_NPZ_PATH_OVERRIDE)
    out["horizon_s"] = np.nan
    if npz_path is None:
        print("No runtime NPZ found; horizon_s left as NaN.")
        return out
    data = np.load(npz_path, allow_pickle=False)
    time_grid = data["time_grid"].astype(float)
    out["horizon_s"] = out["horizon_index"].map(
        lambda idx: float(time_grid[int(idx)]) if 0 <= int(idx) < len(time_grid) else np.nan
    )
    print(f"Loaded time_grid from {npz_path}")
    return out


RESULT_PATH = resolve_result_path(TICKER, MODEL_TYPE, RESULT_PATH_OVERRIDE)
raw_df = pd.read_csv(RESULT_PATH)
df = normalize_grid_frame(raw_df)
df = add_horizon_seconds(df, TICKER)

print(f"Loaded {len(df):,} rows from {RESULT_PATH}")
print(f"Ticker={TICKER}, model_type={MODEL_TYPE}")
print("Grid shape:")
print(df[REQUIRED_GRID_COLS].nunique())
display(df.head())

In [ ]:
def objective_column(df: pd.DataFrame, preferred: str) -> str:
    fallbacks = [preferred, "time_weighted_mean_is_bps", "mean_is_bps", "median_is_bps"]
    for col in fallbacks:
        if col in df.columns and df[col].notna().any():
            return col
    raise ValueError(f"None of the objective columns are available: {fallbacks}")


def feasible_mask(df: pd.DataFrame) -> pd.Series:
    mask = pd.Series(True, index=df.index)
    if "submitted_rate" in df.columns:
        mask &= df["submitted_rate"].fillna(0) >= float(MIN_SUBMITTED_RATE)
    if "metric_failed_rate" in df.columns:
        mask &= df["metric_failed_rate"].fillna(1) <= float(MAX_METRIC_FAILED_RATE)
    if "fill_rate" in df.columns:
        mask &= df["fill_rate"].fillna(0) >= float(MIN_FILL_RATE)
    return mask


def ranking_frame(df: pd.DataFrame, objective: str) -> pd.DataFrame:
    ranked = df.copy()
    ranked["feasible"] = feasible_mask(ranked)
    ranked["objective"] = pd.to_numeric(ranked[objective], errors="coerce")
    ranked["objective_rank"] = ranked["objective"].rank(method="min", ascending=True)
    ranked["selected_rank"] = np.where(ranked["feasible"], ranked["objective_rank"], np.nan)

    tie_break_cols = [
        "feasible",
        "objective",
        "mean_is_bps",
        "submitted_rate",
        "metric_failed_rate",
        "horizon_index",
        "max_toxic_probability",
        "min_fill_probability",
    ]
    tie_break_ascending = [False, True, True, False, True, True, True, True]
    available = [col for col in tie_break_cols if col in ranked.columns]
    ascending = [asc for col, asc in zip(tie_break_cols, tie_break_ascending) if col in ranked.columns]
    ranked = ranked.sort_values(available, ascending=ascending, na_position="last")
    return ranked.reset_index(drop=True)

OBJ = objective_column(df, OBJECTIVE_COL)
ranked = ranking_frame(df, OBJ)

summary_cols = [
    "ticker", "model_type", "horizon_index", "horizon_s",
    "max_toxic_probability", "min_fill_probability", "objective", OBJ,
    "mean_is_bps", "median_is_bps", "time_weighted_mean_is_bps",
    "submitted", "submitted_rate", "fill_rate", "canceled", "canceled_rate",
    "opportunity_cost_mean_bps", "toxic_cost_mean_bps",
    "metric_failed", "metric_failed_rate", "cache_size", "cache_hits", "cache_misses",
]
summary_cols = [c for c in summary_cols if c in ranked.columns]

print(f"Objective: {OBJ} (lower is better)")
print(f"Feasible rows: {int(ranked['feasible'].sum()):,} / {len(ranked):,}")
display(ranked[summary_cols].head(TOP_N))

## Best Configuration by Horizon

This view is usually the cleanest first pass for a 3D grid: it reduces each horizon slice to its best threshold pair, then compares horizons directly.

In [ ]:
best_by_horizon = (
    ranked[ranked["feasible"]]
    .sort_values("objective", ascending=True)
    .groupby("horizon_index", as_index=False)
    .first()
    .sort_values("horizon_index", ascending=True)
)

best_cols = [
    "horizon_index", "horizon_s", "max_toxic_probability", "min_fill_probability",
    "objective", "mean_is_bps", "median_is_bps", "time_weighted_mean_is_bps",
    "submitted_rate", "fill_rate", "canceled_rate", "opportunity_cost_mean_bps", "toxic_cost_mean_bps",
]
best_cols = [c for c in best_cols if c in best_by_horizon.columns]
display(best_by_horizon[best_cols])

fig, ax = plt.subplots(figsize=(8, 4.5))
x = best_by_horizon["horizon_s"] if best_by_horizon["horizon_s"].notna().any() else best_by_horizon["horizon_index"]
ax.plot(x, best_by_horizon["objective"], marker="o")
ax.set_xlabel("Horizon seconds" if best_by_horizon["horizon_s"].notna().any() else "Horizon index")
ax.set_ylabel(f"Best {OBJ}")
ax.set_title("Best feasible strategy by horizon")
ax.grid(True, alpha=0.3)
plt.tight_layout()

## Horizon-Sliced Threshold Heatmaps

Each heatmap is a fixed `horizon_index` slice of the 3D grid. Lower objective values are better.

In [ ]:
def pivot_for_horizon(frame: pd.DataFrame, horizon_index: int, value_col: str) -> pd.DataFrame:
    subset = frame[frame["horizon_index"].eq(int(horizon_index))]
    return subset.pivot_table(
        index="max_toxic_probability",
        columns="min_fill_probability",
        values=value_col,
        aggfunc="mean",
    ).sort_index(ascending=False)


def plot_horizon_heatmaps(frame: pd.DataFrame, value_col: str, *, cmap="RdYlGn_r", fmt=".3f"):
    horizons = sorted(frame["horizon_index"].dropna().astype(int).unique())
    n = len(horizons)
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 4.6), sharey=True)
    if n == 1:
        axes = [axes]

    values = pd.to_numeric(frame[value_col], errors="coerce")
    vmin, vmax = float(values.min()), float(values.max())
    for ax, h in zip(axes, horizons):
        table = pivot_for_horizon(frame, h, value_col)
        label = frame.loc[frame["horizon_index"].eq(h), "horizon_s"].dropna()
        h_title = f"h={h}"
        if not label.empty:
            h_title += f" ({float(label.iloc[0]):.3g}s)"
        sns.heatmap(
            table,
            annot=True,
            fmt=fmt,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            linewidths=0.4,
            cbar=ax is axes[-1],
            ax=ax,
        )
        ax.set_title(h_title)
        ax.set_xlabel("min_fill_probability")
        ax.set_ylabel("max_toxic_probability" if ax is axes[0] else "")
    fig.suptitle(f"{OBJ} by threshold pair and horizon", y=1.03)
    plt.tight_layout()
    return fig, axes

plot_horizon_heatmaps(ranked, OBJ, fmt=".3f");

## Participation and Trade-Off Diagnostics

These plots help detect settings that look good only because they submit too few orders, or because one horizon creates unstable behavior.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

scatter_df = ranked.copy()
size_col = "min_fill_probability"
color_col = "horizon_s" if ranked["horizon_s"].notna().any() else "horizon_index"

sc = axes[0].scatter(
    scatter_df.get("submitted_rate", pd.Series(np.nan, index=scatter_df.index)),
    scatter_df["objective"],
    c=scatter_df[color_col],
    s=55 + 180 * scatter_df[size_col],
    cmap="viridis",
    alpha=0.78,
    edgecolor="black",
    linewidth=0.35,
)
axes[0].set_xlabel("Submitted rate")
axes[0].set_ylabel(OBJ)
axes[0].set_title("Participation vs objective")
axes[0].grid(True, alpha=0.3)
cb = fig.colorbar(sc, ax=axes[0])
cb.set_label(color_col)

if "fill_rate" in scatter_df.columns:
    sns.scatterplot(
        data=scatter_df,
        x="fill_rate",
        y="objective",
        hue="horizon_index",
        size="max_toxic_probability",
        sizes=(35, 180),
        alpha=0.75,
        edgecolor="black",
        linewidth=0.3,
        ax=axes[1],
    )
    axes[1].set_title("Fill rate vs objective")
    axes[1].set_xlabel("Fill rate")
    axes[1].set_ylabel(OBJ)
    axes[1].legend(title="horizon / toxic", bbox_to_anchor=(1.02, 1), loc="upper left")
else:
    axes[1].axis("off")

plt.tight_layout()

## Robustness: Near-Optimal Region

Rather than selecting a single isolated cell, inspect the neighborhood around the best setting. A good final strategy setting should preferably sit in a stable region of the grid.

In [ ]:
best_row = ranked[ranked["feasible"]].iloc[0]
near_tol_bps = 0.05
near = ranked[
    ranked["feasible"]
    & (ranked["objective"] <= float(best_row["objective"]) + near_tol_bps)
].copy()

near_cols = [
    "horizon_index", "horizon_s", "max_toxic_probability", "min_fill_probability",
    "objective", "submitted_rate", "fill_rate", "canceled_rate",
]
near_cols = [c for c in near_cols if c in near.columns]

print(f"Best objective: {float(best_row['objective']):.6f}")
print(f"Rows within {near_tol_bps:g} bps of best: {len(near):,}")
display(near[near_cols].sort_values("objective").head(50))

if not near.empty:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.countplot(data=near, x="horizon_index", color="#4C78A8", ax=ax)
    ax.set_title(f"Near-optimal rows within {near_tol_bps:g} bps of best")
    ax.set_xlabel("horizon_index")
    ax.set_ylabel("count")
    plt.tight_layout()

## Selected Final Strategy Setting

In [ ]:
selected = ranked[ranked["feasible"]].iloc[0]
selected_setting = {
    "ticker": TICKER.upper(),
    "model_type": MODEL_TYPE,
    "max_toxic_probability": float(selected["max_toxic_probability"]),
    "min_fill_probability": float(selected["min_fill_probability"]),
    "horizon_index": int(selected["horizon_index"]),
    "horizon_s": float(selected["horizon_s"]) if pd.notna(selected.get("horizon_s")) else None,
    "objective_col": OBJ,
    "objective_value": float(selected["objective"]),
}

selected_setting


In [ ]:
print("Use this in DeepHitThresholdDecisionLogic:")
print(
    "DeepHitThresholdDecisionLogic("
    f"max_toxic_probability={selected_setting['max_toxic_probability']:.6g}, "
    f"min_fill_probability={selected_setting['min_fill_probability']:.6g}, "
    f"horizon_index={selected_setting['horizon_index']}"
    ")"
)

print()
print("Equivalent CLI args:")
print(
    f"--max-toxic-probability {selected_setting['max_toxic_probability']:.6g} "
    f"--min-fill-probability {selected_setting['min_fill_probability']:.6g} "
    f"--horizon-index {selected_setting['horizon_index']}"
)

In [ ]:
# ---- Selected settings for every ticker/model grid under OUTPUT_ROOT ----
def iter_grid_result_paths(output_root: Path) -> list[Path]:
    return sorted(Path(output_root).glob("*/*/deephit_threshold_grid_search.csv"))


def infer_ticker_model_from_path(path: Path) -> tuple[str, str]:
    return path.parent.parent.name.upper(), path.parent.name


def select_best_setting_for_grid(path: Path) -> dict:
    ticker, model_type = infer_ticker_model_from_path(path)
    raw = pd.read_csv(path)
    frame = normalize_grid_frame(raw)
    frame = add_horizon_seconds(frame, ticker)
    objective = objective_column(frame, OBJECTIVE_COL)
    ranked_frame = ranking_frame(frame, objective)

    feasible = ranked_frame[ranked_frame["feasible"]]
    selected_row = feasible.iloc[0] if len(feasible) else ranked_frame.iloc[0]

    row = {
        "ticker": ticker,
        "model_type": model_type,
        "result_path": str(path),
        "objective_col": objective,
        "selected_feasible": bool(selected_row["feasible"]),
        "objective_value": float(selected_row["objective"]),
        "horizon_index": int(selected_row["horizon_index"]),
        "horizon_s": float(selected_row["horizon_s"]) if pd.notna(selected_row.get("horizon_s")) else np.nan,
        "max_toxic_probability": float(selected_row["max_toxic_probability"]),
        "min_fill_probability": float(selected_row["min_fill_probability"]),
        "feasible_rows": int(ranked_frame["feasible"].sum()),
        "total_rows": int(len(ranked_frame)),
    }

    for col in [
        "mean_is_bps", "median_is_bps", "time_weighted_mean_is_bps",
        "submitted", "submitted_rate", "fill_rate", "canceled_rate",
        "opportunity_cost_mean_bps", "toxic_cost_mean_bps",
        "metric_failed", "metric_failed_rate",
    ]:
        if col in selected_row.index:
            row[col] = selected_row[col]
    return row


result_paths = iter_grid_result_paths(OUTPUT_ROOT)
print(f"Found {len(result_paths):,} grid-search CSV(s) under {OUTPUT_ROOT}")

selected_rows = []
failed_rows = []
for path in result_paths:
    try:
        selected_rows.append(select_best_setting_for_grid(path))
    except Exception as exc:
        ticker, model_type = infer_ticker_model_from_path(path)
        failed_rows.append({
            "ticker": ticker,
            "model_type": model_type,
            "result_path": str(path),
            "error": repr(exc),
        })

all_selected_settings = pd.DataFrame(selected_rows)
if len(all_selected_settings):
    all_selected_settings = all_selected_settings.sort_values(["ticker", "model_type"]).reset_index(drop=True)

    display_cols = [
        "ticker", "model_type", "selected_feasible", "objective_col", "objective_value",
        "horizon_index", "horizon_s", "max_toxic_probability", "min_fill_probability",
        "mean_is_bps", "median_is_bps", "time_weighted_mean_is_bps",
        "submitted_rate", "fill_rate", "canceled_rate", "metric_failed_rate",
        "feasible_rows", "total_rows",
    ]
    display_cols = [c for c in display_cols if c in all_selected_settings.columns]
    display(all_selected_settings[display_cols])
else:
    print("No selectable grid-search results found.")

if failed_rows:
    print(f"Failed to process {len(failed_rows):,} CSV(s):")
    display(pd.DataFrame(failed_rows))



In [ ]:
# ---- Export optimal settings for final evaluation jobs ----
OPTIMAL_SETTINGS_PATH = PROJECT_DIR / "reports" / "final_backtest" / "optimal_strategy_settings.csv"

if "all_selected_settings" not in globals() or all_selected_settings.empty:
    raise RuntimeError(
        "Run the previous all-ticker/model selection cell first so "
        "all_selected_settings is populated."
    )

export_cols = [
    "ticker", "model_type", "selected_feasible", "objective_col", "objective_value",
    "horizon_index", "horizon_s", "max_toxic_probability", "min_fill_probability",
    "mean_is_bps", "median_is_bps", "time_weighted_mean_is_bps",
    "submitted_rate", "fill_rate", "canceled_rate", "metric_failed_rate",
    "feasible_rows", "total_rows", "result_path",
]
export_cols = [col for col in export_cols if col in all_selected_settings.columns]

OPTIMAL_SETTINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
all_selected_settings[export_cols].to_csv(OPTIMAL_SETTINGS_PATH, index=False)
print(f"Wrote {len(all_selected_settings):,} selected setting row(s) to {OPTIMAL_SETTINGS_PATH}")
display(all_selected_settings[export_cols])

